# Notebook 12 — Progetto 1: DocBot

**Settimana:** 12
**Tipo:** Progetto di portfolio
**Tempo stimato:** 5-6 ore

Questo notebook non introduce concetti nuovi: li integra tutti.
Lo usi per verificare che DocBot funzioni end-to-end, costruire il testset
del progetto e generare la scheda delle metriche che mostrerai ai clienti.

## Struttura del progetto

```
proyectos/docbot/
├── app.py              ← interfaccia Streamlit (già creata)
├── docbot/
│   ├── chatbot.py      ← classe DocBot
│   ├── ingesta.py      ← pipeline di ingestione
│   ├── retrieval.py    ← pipeline di query
│   ├── memoria.py      ← sessioni + memoria semantica
│   └── evaluacion.py   ← valutatore di faithfulness
├── requirements.txt
└── README.md
```

## Prima di iniziare

```bash
cd proyectos/docbot
pip install -r requirements.txt
```

Il file `.env` deve trovarsi in `proyectos/docbot/` con la tua `OPENAI_API_KEY`.

---
## Cella 0 — Navigazione alla directory del progetto

In [ ]:
import os
import sys
from pathlib import Path

# Naviga alla directory del progetto affinché gli import funzionino
PROYECTO_DIR = Path("../proyectos/docbot").resolve()
os.chdir(PROYECTO_DIR)
sys.path.insert(0, str(PROYECTO_DIR))

print(f"Directory di lavoro: {os.getcwd()}")
print(f"File del progetto:")
for f in sorted(PROYECTO_DIR.rglob("*.py")):
    print(f"  {f.relative_to(PROYECTO_DIR)}")

---
## Cella 1 — Smoke test: DocBot funziona end-to-end

Verifica che tutti i moduli vengano importati correttamente e che la pipeline
completa funzioni con una domanda semplice.

In [ ]:
from docbot import DocBot, RespuestaDocBot

# ── Inizializzazione ────────────────────────────────────────────────────────
bot = DocBot(
    user_id="test_semana12",
    nueva_sesion=True,       # parte pulito per il test
    evaluar_online=False     # senza valutazione online per lo smoke test
)
print(f"DocBot avviato | Sessione: {bot.session_id}")
print(f"Chunk indicizzati: {bot.n_docs}")

# ── Ingestione del corpus predefinito ────────────────────────────────────────
FUENTES_TEST = [
    {"tipo": "wikipedia", "valor": "FastAPI",               "titulo": "FastAPI",     "categoria": "framework"},
    {"tipo": "wikipedia", "valor": "Pydantic",              "titulo": "Pydantic",    "categoria": "libreria"},
    {"tipo": "wikipedia", "valor": "JSON Web Token",        "titulo": "JWT",         "categoria": "seguridad"},
    {"tipo": "wikipedia", "valor": "Docker (software)",     "titulo": "Docker",      "categoria": "infra"},
    {"tipo": "wikipedia", "valor": "PostgreSQL",            "titulo": "PostgreSQL",  "categoria": "base_datos"},
    {"tipo": "wikipedia", "valor": "Representational state transfer", "titulo": "REST API", "categoria": "concepto"},
]

print("\nAvvio ingestione...")
stats = bot.ingestar(FUENTES_TEST)
print(f"\nStatistiche di ingestione: {stats}")
print(f"Totale chunk nella collezione: {bot.n_docs}")

# ── Smoke test: una domanda ───────────────────────────────────────────────────
print("\nSmoke test — una domanda semplice:")
r = bot.responder("What is FastAPI?")

print(f"  Risposta data: {r.respondida}")
print(f"  Fonti:         {r.fuentes}")
print(f"  Chunk:         {r.n_chunks}")
print(f"  Latenza:       {r.latencia_ms:.0f}ms")
print(f"  Risposta:      {r.respuesta[:200]}...")

assert r.respondida, "ERRORE: avrebbe dovuto rispondere su FastAPI"
assert r.n_chunks > 0, "ERRORE: avrebbe dovuto recuperare dei chunk"
print("\n✓ Smoke test superato.")

---
## Cella 2 — Test di comportamento: i 5 casi che ogni RAG deve superare

Un cliente ti chiederà esattamente questi casi durante la demo.
Verifica che DocBot li gestisca correttamente.

In [ ]:
import json

CASOS_CRITICOS = [
    {
        "id":       "caso_1_responde_bien",
        "tipo":     "risposta corretta",
        "pregunta": "What is Docker and why is it used for application deployment?",
        "espera":   "respondida=True, menziona i container"
    },
    {
        "id":       "caso_2_no_inventa",
        "tipo":     "non allucina",
        "pregunta": "What is the exact revenue of Docker Inc in 2023?",
        "espera":   "respondida=False oppure risposta molto prudente (non può conoscere il revenue)"
    },
    {
        "id":       "caso_3_fuera_corpus",
        "tipo":     "fuori dal corpus",
        "pregunta": "What is the weather like in Madrid today?",
        "espera":   "respondida=False"
    },
    {
        "id":       "caso_4_seguimiento",
        "tipo":     "domanda di follow-up",
        "pregunta": "And what advantages does it have over traditional virtual machines?",
        "espera":   "risponde su Docker (ricorda il contesto precedente)"
    },
    {
        "id":       "caso_5_multi_fuente",
        "tipo":     "multi-fonte",
        "pregunta": "How does FastAPI use Pydantic for data validation?",
        "espera":   "cita sia FastAPI che Pydantic come fonti"
    },
]

print("Esecuzione dei 5 casi critici...\n")
resultados_casos = []

for caso in CASOS_CRITICOS:
    r = bot.responder(caso["pregunta"])
    resultado = {
        "id":        caso["id"],
        "tipo":      caso["tipo"],
        "pregunta":  caso["pregunta"][:60] + "...",
        "respondida": r.respondida,
        "fuentes":   r.fuentes,
        "n_chunks":  r.n_chunks,
        "latencia_ms": r.latencia_ms,
        "respuesta_preview": r.respuesta[:120] + "..."
    }
    resultados_casos.append(resultado)

    icono = "✓" if (caso["id"] in ("caso_1_responde_bien", "caso_4_seguimiento", "caso_5_multi_fuente") and r.respondida) \
               or (caso["id"] in ("caso_2_no_inventa", "caso_3_fuera_corpus") and not r.respondida) \
               else "?"

    print(f"{icono} [{caso['tipo']}]")
    print(f"  D: {caso['pregunta'][:70]}")
    print(f"  R: respondida={r.respondida} | fuentes={r.fuentes} | {r.latencia_ms:.0f}ms")
    print(f"  Atteso: {caso['espera']}")
    print()

# Salva i risultati
with open("evaluaciones/casos_criticos.json", "w") as f:
    json.dump(resultados_casos, f, indent=2, ensure_ascii=False)
print("Risultati salvati in evaluaciones/casos_criticos.json")

**Analisi dei casi critici** (modifica questa cella):

- Caso 2 (non allucina): il sistema ha risposto o rifiutato correttamente?
- Caso 3 (fuori dal corpus): ha restituito il messaggio di "nessuna informazione"?
- Caso 4 (follow-up): ha capito che la domanda riguardava Docker?
- Caso 5 (multi-fonte): ha citato entrambe le fonti?
- Quale caso è fallito o è risultato dubbio?

---
## Cella 3 — Valutazione formale del sistema

Genera un testset con domande rappresentative e valuta faithfulness,
tasso di risposta e rilevanza del contesto.

In [ ]:
import statistics
from docbot.evaluacion import evaluar_turno, resumen_evaluaciones, ResultadoEval
from docbot.retrieval  import recuperar, UMBRAL_RELEVANCIA
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
cliente_eval  = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
chroma_eval   = chromadb.PersistentClient(path="./chroma_db")
coleccion_eval = chroma_eval.get_collection("rag_docs")

# ── Testset di 10 domande rappresentative ─────────────────────────────────────
TESTSET = [
    # Domande dentro il corpus
    "What are the main features of FastAPI?",
    "How does JWT authentication work?",
    "What is the difference between Docker containers and virtual machines?",
    "What is Pydantic used for in Python?",
    "What are the ACID properties of PostgreSQL?",
    "What does REST stand for and what are its constraints?",
    "How does FastAPI handle automatic documentation generation?",
    "What is the history of PostgreSQL?",
    # Domande fuori dal corpus (devono essere rifiutate)
    "What is the current price of Bitcoin?",
    "Who won the Champions League in 2024?",
]

print(f"Valutazione di {len(TESTSET)} domande...")
print("(~3-4 chiamate LLM per domanda — richiederà ~2 minuti)\n")

evaluaciones_testset: list[ResultadoEval] = []

for i, pregunta in enumerate(TESTSET, 1):
    # Genera la risposta
    r = bot.responder(pregunta)

    # Valuta la faithfulness
    eval_r = evaluar_turno(
        pregunta=pregunta,
        respuesta=r.respuesta,
        chunks_textos=[c.texto for c in recuperar(pregunta, coleccion_eval, cliente_eval, k=4) if c.similitud >= UMBRAL_RELEVANCIA],
        chunks_scores=[c.similitud for c in recuperar(pregunta, coleccion_eval, cliente_eval, k=4) if c.similitud >= UMBRAL_RELEVANCIA],
        cliente=cliente_eval
    )
    evaluaciones_testset.append(eval_r)

    faith_str = f"{eval_r.faithfulness:.2f}" if eval_r.faithfulness is not None else "N/A"
    icono     = "✓" if eval_r.respondida else "○"
    print(f"  [{i:2d}] {icono} faithfulness={faith_str:4s} | {pregunta[:55]}...")

# ── Riepilogo delle metriche ───────────────────────────────────────────────────
resumen = resumen_evaluaciones(evaluaciones_testset)

print(f"\n{'='*55}")
print("SCHEDA METRICHE — DocBot v1.0")
print(f"{'='*55}")
print(f"  Domande valutate:        {resumen['n_total']}")
print(f"  Tasso di risposta:       {resumen['tasa_respuesta']:.0%}")
print(f"  Faithfulness media:      {resumen.get('faithfulness_media', 'N/A')}")
print(f"  Context relevance:       {resumen.get('context_rel_media', 'N/A')}")
print(f"  Risposte con allucin.:   {resumen.get('n_con_alucinaciones', 0)}")
print(f"  Distribuzione verdetti:  {resumen.get('distribucion_veredictos', {})}")
print(f"{'='*55}")

# Criterio di superamento
faith = resumen.get('faithfulness_media')
tasa  = resumen['tasa_respuesta']
aprobado = faith is not None and faith >= 0.80 and tasa >= 0.70
print(f"\n{'✓ PROGETTO SUPERATO' if aprobado else '✗ PROGETTO DA MIGLIORARE'}")
if not aprobado:
    if faith is not None and faith < 0.80:
        print(f"  → Faithfulness {faith:.2f} < 0.80: rafforza il grounding nel prompt")
    if tasa < 0.70:
        print(f"  → Tasso di risposta {tasa:.0%} < 70%: abbassa la soglia di rilevanza o aumenta k")

# Salva la scheda delle metriche
import json
with open("evaluaciones/tarjeta_metricas_v1.json", "w") as f:
    json.dump({"version": "1.0", "fecha": __import__('datetime').date.today().isoformat(), **resumen}, f, indent=2)
print("\nScheda salvata in evaluaciones/tarjeta_metricas_v1.json")

---
## Cella 4 — Test della memoria tra sessioni

Verifica che la memoria semantica funzioni: chiudi la sessione,
aprine una nuova e controlla che il bot ricordi l'utente.

In [ ]:
# ── Sessione 1: l'utente si presenta ──────────────────────────────────────────
print("SESSIONE 1: l'utente si presenta")
print("=" * 55)

bot_s1 = DocBot(user_id="raffaele_portfolio", nueva_sesion=True)

mensajes_s1 = [
    "Hi, I'm Raffaele. I work at DIH DATAlife, an innovation hub in Galicia, Spain.",
    "I'm specializing in AI Engineering for the agri-food and biotech sector.",
    "What is Docker and could it be useful for deploying AI applications?",
]

for msg in mensajes_s1:
    r = bot_s1.responder(msg)
    print(f"\nTu: {msg}")
    print(f"Bot: {r.respuesta[:180]}..." if len(r.respuesta) > 180 else f"Bot: {r.respuesta}")

print(f"\nChiusura sessione 1...")
bot_s1.cerrar_sesion()
print(f"Sessione 1 ({bot_s1.session_id}) chiusa.")

# ── Sessione 2: il bot ricorda l'utente? ──────────────────────────────────────
print("\n\nSESSIONE 2: nuova sessione — il bot ricorda?")
print("=" * 55)

bot_s2 = DocBot(user_id="raffaele_portfolio")  # carica la sessione più recente

mensajes_s2 = [
    "Hey, I'm back! Do you remember anything about me?",
    "Given my domain (agri-food and biotech), would you recommend FastAPI or Django REST?",
]

for msg in mensajes_s2:
    r = bot_s2.responder(msg)
    print(f"\nTu: {msg}")
    print(f"Bot: {r.respuesta[:250]}..." if len(r.respuesta) > 250 else f"Bot: {r.respuesta}")

# ── Verifica ────────────────────────────────────────────────────────────────
primera_respuesta = bot_s2.responder("Do you remember where I work?")
menciona_datalife = any(kw in primera_respuesta.respuesta.lower()
                        for kw in ["datalife", "galicia", "innovation", "hub", "spain"])

print(f"\n{'='*55}")
print(f"Test memoria: {'✓ SUPERATO' if menciona_datalife else '✗ FALLITO'}")
print(f"  La risposta {'menziona' if menciona_datalife else 'NON menziona'} dati della sessione precedente")
bot_s2.cerrar_sesion()

---
## Cella 5 — Avviare l'interfaccia Streamlit

Quando il codice supera tutti i test precedenti, avvia l'interfaccia.
Usala per fare la demo al cliente.

In [ ]:
# Istruzioni per avviare l'app
print("Per avviare DocBot come applicazione web:")
print()
print("  cd proyectos/docbot")
print("  streamlit run app.py")
print()
print("Apri http://localhost:8501 nel tuo browser.")
print()
print("Checklist pre-demo:")
print("  [ ] L'app si avvia senza errori")
print("  [ ] L'ingestione di 3+ documenti funziona")
print("  [ ] La chat risponde a domande sui documenti")
print("  [ ] Le domande fuori dal corpus mostrano il messaggio 'nessuna informazione'")
print("  [ ] Lo storico persiste al refresh della pagina")
print("  [ ] La scheda delle metriche mostra faithfulness ≥ 0.80")

---
## Sfida: adatta il corpus al tuo dominio

Il corpus di Wikipedia è il fallback gratuito. Perché questo progetto
abbia un vero valore di portfolio, adattalo al tuo dominio (agri-food, biotech, forestale).

**Opzione A — Wikipedia in italiano/inglese sul tuo dominio:**

In [ ]:
# Corpus di dominio: agri-food e biotech (rilevante per DIH DATAlife)
# Decommenta ed esegui per indicizzare un corpus più rappresentativo

# CORPUS_DATALIFE = [
#     # Agri-food
#     {"tipo": "wikipedia", "valor": "Precision agriculture",         "titulo": "Precision Agriculture",   "categoria": "agri-food"},
#     {"tipo": "wikipedia", "valor": "Food traceability",             "titulo": "Food Traceability",        "categoria": "agri-food"},
#     {"tipo": "wikipedia", "valor": "Aquaculture",                   "titulo": "Aquaculture",              "categoria": "agri-food"},
#     {"tipo": "wikipedia", "valor": "Common Fisheries Policy",       "titulo": "Common Fisheries Policy",  "categoria": "agri-food"},
#     # Biotech
#     {"tipo": "wikipedia", "valor": "Bioinformatics",                "titulo": "Bioinformatics",           "categoria": "biotech"},
#     {"tipo": "wikipedia", "valor": "CRISPR",                        "titulo": "CRISPR",                   "categoria": "biotech"},
#     {"tipo": "wikipedia", "valor": "Synthetic biology",             "titulo": "Synthetic Biology",        "categoria": "biotech"},
#     # Forestale
#     {"tipo": "wikipedia", "valor": "Sustainable forest management", "titulo": "Sustainable Forestry",    "categoria": "forestal"},
#     {"tipo": "wikipedia", "valor": "Forest Stewardship Council",    "titulo": "FSC Certification",        "categoria": "forestal"},
# ]

# bot_dominio = DocBot(user_id="docbot_datalife", coleccion_docs="rag_datalife", nueva_sesion=True)
# stats = bot_dominio.ingestar(CORPUS_DATALIFE)
# print(stats)

**Opzione B — I tuoi documenti in Markdown:**

```python
# Metti i tuoi documenti in proyectos/docbot/datos/
# Esempio: guide tecniche, report, normativa

from pathlib import Path

mis_docs = [
    {"tipo": "markdown", "valor": str(p), "titulo": p.stem, "categoria": "interno"}
    for p in Path("datos").glob("*.md")
]
print(f"{len(mis_docs)} documenti trovati in datos/")

if mis_docs:
    bot_propio = DocBot(user_id="mi_usuario", coleccion_docs="rag_propio", nueva_sesion=True)
    stats = bot_propio.ingestar(mis_docs)
    print(stats)
```

---
## Riflessione post-progetto

Dedica 30 minuti a rispondere a queste domande prima di passare alla settimana 13.

1. **Qual è stata la parte più difficile da integrare?**
   ...

2. **Quale domanda del testset è fallita più spesso e perché?**
   ...

3. **Se un cliente del settore agri-food ti chiedesse questo sistema, quanto gli faresti pagare e in quanto tempo lo consegneresti?**
   ...

4. **Quale funzionalità aggiungeresti per prima con una settimana extra?**
   ...

5. **Quale parte del codice riutilizzeresti direttamente per il Progetto 2?**
   ...

---
## Chiusura del blocco Agenti (S9-S12)

Hai completato le quattro settimane del blocco:

- **S09** Agenti ReAct con strumenti
- **S10** Memoria e stato tra sessioni
- **S11** Pianificazione e sistemi multi-agente
- **S12** Progetto 1: DocBot ← qui

La settimana 13 inizia il **Blocco 6: Deployment e produzione** —
come trasformare DocBot in un'API REST con FastAPI, dockerizzarla
e distribuirla nel cloud.

---
*Notebook 12 — Settimana 12 del Piano di Formazione AI Engineering*